# YOLOv8 目标检测 — 本地GPU训练
RTX 3060 6GB, PyTorch 2.4.1+cu124, ultralytics 8.4.60

训练完成后将模型上传至云端 Ascend NPU 进行推理对比

In [1]:
import torch
from ultralytics import YOLO
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"GPU:     {torch.cuda.get_device_name(0)}")
print(f"显存:    {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch: 2.4.1+cu124
CUDA:    True
GPU:     NVIDIA GeForce RTX 3060 Laptop GPU
显存:    6.0 GB


## 1. 用 COCO128 小数据集训练 YOLOv8n

In [ ]:
# 使用本地文件，避免在线下载
import os, torch
torch.multiprocessing.set_sharing_strategy('file_system')
WORK_DIR = r'd:\codeproject\shengteng\workfile'
os.chdir(WORK_DIR)
print(f'工作目录: {os.getcwd()}')

model = YOLO('yolov8n.pt')

results = model.train(
    data='coco128/coco128.yaml',
    epochs=10,
    imgsz=320,
    batch=1,
    device=0,
    workers=0,
    project='runs/yolov8_local',
    name='train',
    exist_ok=True
)

工作目录: d:\codeproject\shengteng\workfile
Ultralytics 8.4.60  Python-3.8.20 torch-2.4.1+cu124 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco128/coco128.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, 

## 2. 验证训练结果

In [ ]:
# 在验证集上评估
metrics = model.val()
print(f"\nmAP@50:      {metrics.box.map50:.3f}")
print(f"mAP@50-95:   {metrics.box.map:.3f}")

## 3. 单张图片推理测试

In [ ]:
# 用训练好的模型跑推理
model = YOLO('runs/yolov8_local/train/weights/best.pt')

# 下载一张测试图片
import urllib.request
urllib.request.urlretrieve(
    'https://ultralytics.com/images/bus.jpg',
    'test_bus.jpg'
)

results = model('test_bus.jpg', device=0)
results[0].show()  # 弹出窗口显示检测结果

## 4. GPU 推理速度基准测试

In [ ]:
import time

model = YOLO('runs/yolov8_local/train/weights/best.pt')

# 预热
for _ in range(5):
    _ = model('test_bus.jpg', device=0, verbose=False)
torch.cuda.synchronize()

# 计时
t0 = time.time()
for _ in range(50):
    _ = model('test_bus.jpg', device=0, verbose=False)
torch.cuda.synchronize()
gpu_time = (time.time() - t0) / 50 * 1000

print(f"GPU (RTX 3060) 单张推理: {gpu_time:.1f} ms")
print(f"GPU (RTX 3060) FPS:       {1000/gpu_time:.1f}")

## 5. 导出模型（为 NPU 迁移准备）

In [ ]:
# 保存权重文件路径，后续云端 NPU 用 torch.load 直接加载
best_pt = 'runs/yolov8_local/train/weights/best.pt'

# 导出为 ONNX（后续可用 ATC 转换为 .om 离线模型）
model.export(format='onnx', imgsz=640, simplify=True)

print(f"\n本地文件:")
print(f"  最佳权重: {best_pt}")
print(f"  ONNX:     runs/yolov8_local/train/weights/best.onnx")